# 04 — Salary Trends 2020–2025

Use aijobs.net to show salary trends, remote ratio evolution, and company size effects across years.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
os.chdir("/app")
import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_PROC = pathlib.Path("data/processed")


In [2]:
aijobs = pd.read_parquet(DATA_PROC / "aijobs_processed.parquet")
print(f"Shape: {aijobs.shape}")
print(f"Years: {sorted(aijobs['work_year'].unique())}")
print(f"Rows per year:\n{aijobs['work_year'].value_counts().sort_index().to_string()}")
print(f"\nTop 15 job titles:")
print(aijobs["job_title"].value_counts().head(15).to_string())
print(f"\nExperience levels: {aijobs['experience_level'].unique()}")
print(f"Remote ratio values: {sorted(aijobs['remote_ratio'].unique())}")
print(f"Company size values: {aijobs['company_size'].unique()}")


Shape: (151445, 11)
Years: [2020, 2021, 2022, 2023, 2024, 2025]
Rows per year:
work_year
2020       75
2021      218
2022     1661
2023     8524
2024    62241
2025    78726

Top 15 job titles:
job_title
Data Scientist               18751
Software Engineer            16948
Data Engineer                16352
Data Analyst                 13779
Engineer                     11004
Machine Learning Engineer     8887
Manager                       7811
Analyst                       5396
Research Scientist            3460
Product Manager               2576
Applied Scientist             2381
Associate                     2379
Data Architect                2216
Analytics Engineer            2139
AI Engineer                   2013

Experience levels: ['EX' 'SE' 'MI' 'EN']
Remote ratio values: [0, 50, 100]
Company size values: ['M' 'L' 'S']


## Median Salary by Year for Top 10 Roles

In [3]:
# Focus on full-time and sufficient history
ai_ft = aijobs[aijobs["employment_type"] == "FT"].copy()

# Top 10 roles by total count (across all years)
top10_roles = ai_ft["job_title"].value_counts().head(10).index.tolist()
print("Top 10 roles:", top10_roles)

trend_data = (
    ai_ft[ai_ft["job_title"].isin(top10_roles)]
    .groupby(["work_year", "job_title"])["salary_in_usd"]
    .median()
    .reset_index()
    .rename(columns={"salary_in_usd": "median_salary"})
)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(top10_roles)))
for i, role in enumerate(top10_roles):
    d = trend_data[trend_data["job_title"] == role].sort_values("work_year")
    if len(d) >= 3:
        ax.plot(d["work_year"], d["median_salary"] / 1_000, marker="o", label=role[:30], color=colors[i])
ax.set_xlabel("Year")
ax.set_ylabel("Median Annual Salary ($K)")
ax.set_title("Salary Trend by Role — aijobs.net 2020-2025")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("figures/fig_salary_trends.png", dpi=120, bbox_inches="tight")
plt.show()


Top 10 roles: ['Data Scientist', 'Software Engineer', 'Data Engineer', 'Data Analyst', 'Engineer', 'Machine Learning Engineer', 'Manager', 'Analyst', 'Research Scientist', 'Product Manager']


## Remote Work Ratio Trend

In [4]:
remote_trend = (
    ai_ft.groupby("work_year")["remote_ratio"]
    .mean()
    .reset_index()
    .rename(columns={"remote_ratio": "avg_remote_ratio"})
)
print("Remote ratio trend:")
print(remote_trend.to_string())

# Also show breakdown by remote_ratio category
remote_cat_trend = (
    ai_ft.groupby(["work_year", "remote_ratio"])
    .size()
    .reset_index(name="count")
)
total_per_year = ai_ft.groupby("work_year").size().reset_index(name="total")
remote_cat_trend = remote_cat_trend.merge(total_per_year, on="work_year")
remote_cat_trend["pct"] = 100 * remote_cat_trend["count"] / remote_cat_trend["total"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(remote_trend["work_year"], remote_trend["avg_remote_ratio"], marker="o", color="#4C72B0")
ax1.set_title("Average Remote Ratio by Year")
ax1.set_xlabel("Year")
ax1.set_ylabel("Average Remote Ratio (0=onsite, 100=fully remote)")

remote_labels = {0: "Onsite (0%)", 50: "Hybrid (50%)", 100: "Fully Remote (100%)"}
cat_colors = {0: "#DD4444", 50: "#DDAA00", 100: "#44AA44"}
for ratio_val, color in cat_colors.items():
    d = remote_cat_trend[remote_cat_trend["remote_ratio"] == ratio_val].sort_values("work_year")
    ax2.plot(d["work_year"], d["pct"], marker="o", label=remote_labels[ratio_val], color=color)
ax2.set_title("Work Arrangement Mix by Year (%)")
ax2.set_xlabel("Year")
ax2.set_ylabel("% of Job Postings")
ax2.legend()
plt.tight_layout()
plt.savefig("figures/fig_remote_trends.png", dpi=120, bbox_inches="tight")
plt.show()


Remote ratio trend:
   work_year  avg_remote_ratio
0       2020         61.594203
1       2021         69.711538
2       2022         55.039466
3       2023         31.726813
4       2024         19.041277
5       2025         20.244873


## Company Size Effect on Salary

In [5]:
size_trend = (
    ai_ft.groupby(["work_year", "company_size"])["salary_in_usd"]
    .median()
    .reset_index()
    .rename(columns={"salary_in_usd": "median_salary"})
)
size_labels = {"S": "Small (<50)", "M": "Medium (50-250)", "L": "Large (>250)"}
size_colors = {"S": "#4C72B0", "M": "#55A868", "L": "#DD8452"}

fig, ax = plt.subplots(figsize=(9, 5))
for size, color in size_colors.items():
    d = size_trend[size_trend["company_size"] == size].sort_values("work_year")
    ax.plot(d["work_year"], d["median_salary"] / 1_000, marker="o",
            label=size_labels.get(size, size), color=color, linewidth=2)
ax.set_xlabel("Year")
ax.set_ylabel("Median Salary ($K)")
ax.set_title("Median Salary by Company Size — aijobs.net")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_company_size_trend.png", dpi=120, bbox_inches="tight")
plt.show()

print("Company size stats by year:")
print(size_trend.pivot(index="work_year", columns="company_size", values="median_salary")
      .div(1000).round(1).to_string())


Company size stats by year:
company_size      L      M     S
work_year                       
2020           89.0  103.5  62.7
2021           93.8   78.8  80.5
2022          125.0  135.0  65.0
2023          136.0  146.0  80.0
2024          150.4  149.0  94.4
2025          151.3  145.5  56.8


## Overall Salary Trend (All Roles)

In [6]:
overall_trend = (
    ai_ft.groupby("work_year")["salary_in_usd"]
    .agg(median="median", p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75), count="count")
    .reset_index()
)
print("Overall salary trend:")
print(overall_trend.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(overall_trend["work_year"],
                overall_trend["p25"] / 1_000,
                overall_trend["p75"] / 1_000,
                alpha=0.25, color="#4C72B0", label="P25–P75 band")
ax.plot(overall_trend["work_year"], overall_trend["median"] / 1_000,
        marker="o", color="#4C72B0", linewidth=2, label="Median")
ax.set_xlabel("Year")
ax.set_ylabel("Salary ($K)")
ax.set_title("Overall Salary Trend (All Roles) — aijobs.net")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_overall_trend.png", dpi=120, bbox_inches="tight")
plt.show()


Overall salary trend:
   work_year    median       p25       p75  count
0       2020   87000.0   49268.0  120000.0     69
1       2021   86369.0   54202.0  140000.0    208
2       2022  132100.0   95000.0  172347.5   1647
3       2023  145000.0  109400.0  190000.0   8507
4       2024  149200.0  108191.0  200000.0  61947
5       2025  145900.0  105000.0  197200.0  78163


## Save Results

In [7]:
# Save salary_trends.parquet
salary_trends = trend_data.copy()
salary_trends.to_parquet(DATA_PROC / "salary_trends.parquet", index=False)

# Save remote trend
remote_trend.to_parquet(DATA_PROC / "remote_trend.parquet", index=False)

# Save company size trend
size_trend.to_parquet(DATA_PROC / "company_size_trend.parquet", index=False)

# Save overall trend
overall_trend.to_parquet(DATA_PROC / "overall_salary_trend.parquet", index=False)

print(f"Saved salary_trends.parquet — {len(salary_trends)} rows ({len(top10_roles)} roles)")
print(f"Saved remote_trend.parquet")
print(f"Saved company_size_trend.parquet")
print(f"Saved overall_salary_trend.parquet")


Saved salary_trends.parquet — 40 rows (10 roles)
Saved remote_trend.parquet
Saved company_size_trend.parquet
Saved overall_salary_trend.parquet
